# Contribuição individual — Entrega 06: Bianca Lavine

Este notebook registra o bloco individual de Bianca para posterior consolidação no notebook coletivo da E6. Ele cobre a entrada comum E4/E5, os cortes temporais, o baseline D-7 como heurística proposta e a regressão Ridge. O notebook final da E4 é mantido sem alterações.

Grão: data + tipo de refeição + curso. Alvo: quantidade servida. O total do RU é derivado dos cursos quando a cobertura está completa. Dados: mock sintético.

## 4. Dados e protocolo comum

A base é refeita dos CSVs do mock pela mesma regra consolidada: contar IDs únicos de eventos servidos, montar a grade de data/refeição/curso e preencher ausência com zero somente sob cobertura completa. A chave analítica combina data, refeição e curso; o ID do evento é usado apenas para contar.

Preditoras conhecidas na véspera: dia da semana, mês, tipo de refeição e curso. Alvo: quantidade servida. Aberto e extração completa são filtros, não preditoras; cadastro de cursos não alimenta o modelo. Cortes: treino até 31/12/2024; validação 2025; teste a partir de 01/01/2026, reservado.

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Acha a raiz do repositório mesmo se o notebook for aberto em notebooks/.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados/bruto/mock").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório.")
BRUTOS = RAIZ / "dados/bruto/mock"
catraca = pd.read_csv(BRUTOS / "catraca_ru.csv", dtype="string")
estudantes = pd.read_csv(BRUTOS / "estudantes.csv", dtype="string")
catraca["data"] = pd.to_datetime(catraca["data_hora_refeicao"], errors="coerce").dt.normalize()
if catraca["data"].isna().any() or catraca["id_registro_utilizacao"].duplicated().any():
    raise ValueError("Data inválida ou identificador de evento duplicado.")
servidas = catraca.loc[catraca["situacao_evento"].eq("servida")].copy()
chave = ["data", "tipo_refeicao", "curso"]
contagens = (servidas.groupby(chave)["id_registro_utilizacao"].nunique()
             .reset_index(name="quantidade_servida"))

# Reproduz a grade/cobertura e a regra de zero da preparação coletiva da E4.
datas = pd.date_range("2019-01-01", "2026-11-14", freq="D")
refeicoes = sorted(catraca["tipo_refeicao"].dropna().unique())
cursos = sorted(estudantes["curso"].dropna().unique())
cobertura = pd.MultiIndex.from_product([datas, refeicoes, cursos], names=chave).to_frame(index=False)
cobertura["aberto"] = cobertura["data"].dt.day_of_week.lt(6)
cobertura["extracao_completa"] = True
base = cobertura.merge(contagens, how="left", on=chave, validate="one_to_one")
if base.loc[~base["aberto"], "quantidade_servida"].notna().any():
    raise ValueError("O mock tem eventos em dia marcado como fechado.")
base["quantidade_servida"] = base["quantidade_servida"].fillna(0).astype("int64")
assert int(base["quantidade_servida"].sum()) == len(servidas)
modelavel = base.loc[base["aberto"] & base["extracao_completa"]].copy()

# Cortes comuns da E4/E5: treino até 2024, validação em 2025, teste em 2026.
FIM_TREINO, FIM_VALIDACAO = pd.Timestamp("2024-12-31"), pd.Timestamp("2025-12-31")
treino = modelavel.loc[modelavel["data"].le(FIM_TREINO)].copy()
validacao = modelavel.loc[modelavel["data"].gt(FIM_TREINO) & modelavel["data"].le(FIM_VALIDACAO)].copy()
teste = modelavel.loc[modelavel["data"].gt(FIM_VALIDACAO)].copy()
assert treino["data"].max() < validacao["data"].min() < teste["data"].min()

def entradas(t):
    return pd.DataFrame({
        "dia_semana": t["data"].dt.day_of_week.to_numpy(),
        "mes": t["data"].dt.month.to_numpy(),
        "tipo_refeicao": t["tipo_refeicao"].to_numpy(),
        "curso": t["curso"].to_numpy(),
    }, index=t.index)
Xtr, Xva = entradas(treino), entradas(validacao)
ytr, yva = treino["quantidade_servida"].astype("int64"), validacao["quantidade_servida"].astype("int64")

def pipeline(alpha):
    prep = ColumnTransformer([
        ("num", Pipeline([("imputa", SimpleImputer(strategy="median")),
                          ("escala", StandardScaler())]), ["dia_semana", "mes"]),
        ("cat", Pipeline([("imputa", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
         ["tipo_refeicao", "curso"]),
    ])
    return Pipeline([("preprocessador", prep), ("ridge", Ridge(alpha=alpha))])

# Dobras por datas únicas; todas as linhas do mesmo dia ficam juntas.
datas_treino = np.sort(treino["data"].drop_duplicates().to_numpy())
dobras = []
for ifit, iva in TimeSeriesSplit(n_splits=5).split(datas_treino):
    df, dv = datas_treino[ifit], datas_treino[iva]
    mf, mv = treino["data"].isin(df), treino["data"].isin(dv)
    assert set(df).isdisjoint(set(dv)) and df.max() < dv.min()
    dobras.append((mf, mv, df, dv))

# Grid pequena; escolher alpha somente pelo MAE médio do treino em CV.
alphas = [0.1, 1.0, 10.0, 100.0]
registros = []
t_busca = time.perf_counter()
for alpha in alphas:
    for n, (mf, mv, df, dv) in enumerate(dobras, start=1):
        modelo = pipeline(alpha)
        t0 = time.perf_counter()
        modelo.fit(Xtr.loc[mf], ytr.loc[mf])
        seg = time.perf_counter() - t0
        pred, real = modelo.predict(Xtr.loc[mv]), ytr.loc[mv]
        registros.append({"alpha":alpha, "fold":n, "inicio_fit":str(pd.Timestamp(df.min()).date()),
            "fim_fit":str(pd.Timestamp(df.max()).date()), "inicio_validacao":str(pd.Timestamp(dv.min()).date()),
            "fim_validacao":str(pd.Timestamp(dv.max()).date()), "linhas_fit":int(mf.sum()),
            "linhas_validacao":int(mv.sum()), "MAE":mean_absolute_error(real,pred),
            "RMSE":np.sqrt(mean_squared_error(real,pred)), "tempo_fit_segundos":seg})
tempo_busca = time.perf_counter() - t_busca
cv_detalhe = pd.DataFrame(registros)
cv_resumo = cv_detalhe.groupby("alpha").agg(MAE_medio=("MAE","mean"), RMSE_medio=("RMSE","mean"),
    tempo_fit_cv_segundos=("tempo_fit_segundos","sum")).reset_index()
alpha_escolhido = float(cv_resumo.sort_values(["MAE_medio","alpha"]).iloc[0]["alpha"])

# Ajuste final somente no treino; validação recebe previsões; teste fica sem uso.
modelo_linear = pipeline(alpha_escolhido)
t0 = time.perf_counter()
modelo_linear.fit(Xtr, ytr)
tempo_fit_final = time.perf_counter() - t0
pred_ridge = modelo_linear.predict(Xva)

# D-7 é heurística proposta, não regra atual confirmada do RU.
anterior = modelavel[["data","tipo_refeicao","curso","quantidade_servida"]].copy()
anterior["data"] += pd.Timedelta(days=7)
anterior = anterior.rename(columns={"quantidade_servida":"previsao_d7"})
predicoes = validacao[["data","tipo_refeicao","curso","quantidade_servida"]].copy()
predicoes["previsao_ridge"] = pred_ridge
predicoes = predicoes.merge(anterior, how="left", on=["data","tipo_refeicao","curso"], validate="one_to_one")
predicoes["id_linha"] = predicoes.data.dt.strftime("%Y-%m-%d")+"|"+predicoes.tipo_refeicao+"|"+predicoes.curso
predicoes["modelo_linear"] = f"Ridge(alpha={alpha_escolhido:g})"
predicoes["baseline"] = "D-7 heuristica proposta"

def metricas(y,p):
    return {"linhas":int(len(y)), "MAE":float(mean_absolute_error(y,p)),
            "RMSE":float(np.sqrt(mean_squared_error(y,p)))}
m_ridge = metricas(predicoes.quantidade_servida,predicoes.previsao_ridge)
com_d7 = predicoes.dropna(subset=["previsao_d7"])
m_d7 = metricas(com_d7.quantidade_servida,com_d7.previsao_d7)

# Total RU derivado apenas se todos os cursos estiverem presentes.
totais = predicoes.groupby(["data","tipo_refeicao"]).agg(cursos_presentes=("curso","nunique"),
    quantidade_real_ru=("quantidade_servida","sum"), previsao_ridge_ru=("previsao_ridge","sum"),
    previsao_d7_ru=("previsao_d7","sum")).reset_index()
totais = totais.loc[totais.cursos_presentes.eq(len(cursos))].copy()
totais["id_linha"] = totais.data.dt.strftime("%Y-%m-%d")+"|"+totais.tipo_refeicao+"|RU_TOTAL"

# Formato comum de previsões: ID, data, refeição, curso, partição, modelo, observado e previsto.
pred_larga=predicoes.copy()
predicoes=pd.concat([
 pd.DataFrame({"id_observacao":pred_larga.id_linha,"data":pred_larga.data,
  "tipo_refeicao":pred_larga.tipo_refeicao,"curso":pred_larga.curso,"particao":"validacao_2025",
  "modelo":"Ridge","valor_observado":pred_larga.quantidade_servida,
  "valor_previsto":pred_larga.previsao_ridge,"parametros":f"alpha={alpha_escolhido:g}"}),
 pd.DataFrame({"id_observacao":pred_larga.id_linha,"data":pred_larga.data,
  "tipo_refeicao":pred_larga.tipo_refeicao,"curso":pred_larga.curso,"particao":"validacao_2025",
  "modelo":"D-7 heuristica proposta","valor_observado":pred_larga.quantidade_servida,
  "valor_previsto":pred_larga.previsao_d7,"parametros":"sem ajuste"})
],ignore_index=True)
tot_largo=totais.copy()
totais=pd.concat([
 pd.DataFrame({"id_observacao":tot_largo.id_linha,"data":tot_largo.data,
  "tipo_refeicao":tot_largo.tipo_refeicao,"curso":"RU_TOTAL","particao":"validacao_2025",
  "modelo":"Ridge","valor_observado":tot_largo.quantidade_real_ru,
  "valor_previsto":tot_largo.previsao_ridge_ru,"cursos_presentes":tot_largo.cursos_presentes,
  "parametros":f"alpha={alpha_escolhido:g}"}),
 pd.DataFrame({"id_observacao":tot_largo.id_linha,"data":tot_largo.data,
  "tipo_refeicao":tot_largo.tipo_refeicao,"curso":"RU_TOTAL","particao":"validacao_2025",
  "modelo":"D-7 heuristica proposta","valor_observado":tot_largo.quantidade_real_ru,
  "valor_previsto":tot_largo.previsao_d7_ru,"cursos_presentes":tot_largo.cursos_presentes,
  "parametros":"sem ajuste"})
],ignore_index=True)
assert len(predicoes)==2*len(validacao) and predicoes.id_observacao.nunique()==len(validacao)
SAIDA = RAIZ/"dados/saidas/e06_bianca"
SAIDA.mkdir(parents=True,exist_ok=True)
cv_detalhe.to_csv(SAIDA/"validacao_cruzada_ridge_por_dobra.csv",index=False)
cv_resumo.to_csv(SAIDA/"resumo_validacao_cruzada_ridge.csv",index=False)
predicoes.to_csv(SAIDA/"previsoes_validacao_bianca.csv",index=False)
totais.to_csv(SAIDA/"previsoes_total_ru_validacao_bianca.csv",index=False)
execucao = pd.DataFrame([
 {"metodo":"Ridge regularizado","perda_selecao":"MAE medio em 5 dobras temporais do treino",
  "alpha":alpha_escolhido,"MAE_validacao_2025":m_ridge["MAE"],"RMSE_validacao_2025":m_ridge["RMSE"],
  "linhas_validacao":m_ridge["linhas"],"tempo_busca_segundos":tempo_busca,
  "tempo_fit_final_segundos":tempo_fit_final,"semente":"não aplicável (Ridge determinístico)"},
 {"metodo":"D-7 heuristica proposta","perda_selecao":"sem ajuste de parametro","alpha":np.nan,
  "MAE_validacao_2025":m_d7["MAE"],"RMSE_validacao_2025":m_d7["RMSE"],
  "linhas_validacao":m_d7["linhas"],"tempo_busca_segundos":0.0,
  "tempo_fit_final_segundos":0.0,"semente":"não aplicável"}])
execucao.to_csv(SAIDA/"registro_execucao_e06_bianca.csv",index=False)
print(f"Eventos servidos/soma base: {len(servidas):,}/{base.quantidade_servida.sum():,}")
print(f"Grão: data+refeição+curso | alvo: quantidade_servida | cursos: {len(cursos)}")
print(f"Treino: {len(treino):,} linhas ({treino.data.min().date()} a {treino.data.max().date()})")
print(f"Validação: {len(validacao):,} linhas em 2025 | teste reservado: {len(teste):,} linhas em 2026")
print("Resumo CV por alpha; seleção pelo MAE médio:")
print(cv_resumo.round(4).to_string(index=False))
print(f"Alpha escolhido={alpha_escolhido:g}; busca={tempo_busca:.2f}s; ajuste final={tempo_fit_final:.2f}s")
print("Métricas validação:", pd.DataFrame([{"modelo":"Ridge",**m_ridge},{"modelo":"D-7 proposto",**m_d7}]).round(3).to_dict("records"))
print(f"D-7 com histórico: {len(com_d7):,}/{len(validacao):,}; totais RU derivados: {len(totais):,}")
print("Teste 2026 não foi usado para ajustar, escolher parâmetro ou calcular métricas.")
print("Saídas:",SAIDA)


Eventos servidos/soma base: 221,600/221,600
Grão: data+refeição+curso | alvo: quantidade_servida | cursos: 6
Treino: 33,822 linhas (2019-01-01 a 2024-12-31)
Validação: 5,634 linhas em 2025 | teste reservado: 4,914 linhas em 2026
Resumo CV por alpha; seleção pelo MAE médio:
 alpha  MAE_medio  RMSE_medio  tempo_fit_cv_segundos
   0.1     2.2638      2.8480                 0.1268
   1.0     2.2639      2.8480                 0.0769
  10.0     2.2643      2.8480                 0.0771
 100.0     2.2694      2.8486                 0.0792
Alpha escolhido=0.1; busca=0.49s; ajuste final=0.02s
Métricas validação: [{'modelo': 'Ridge', 'linhas': 5634, 'MAE': 2.439, 'RMSE': 3.038}, {'modelo': 'D-7 proposto', 'linhas': 5634, 'MAE': 2.139, 'RMSE': 2.951}]
D-7 com histórico: 5,634/5,634; totais RU derivados: 1,878
Teste 2026 não foi usado para ajustar, escolher parâmetro ou calcular métricas.
Saídas: C:\Users\biala\.codex\worktrees\e06-lavine\previsao-demanda-ru-ufpb\dados\saidas\e06_bianca


## 12. Reprodutibilidade, seleção e resultados

A validação cruzada tem cinco dobras cronológicas construídas sobre datas únicas, mantendo as linhas de cada dia juntas. Imputação, padronização, codificação e Ridge ficam em um Pipeline reajustado em cada dobra. A grade de alpha é 0,1; 1; 10; 100. O parâmetro é escolhido pelo menor MAE médio no treino; RMSE é complementar. O teste de 2026 não participa da escolha.

O ajuste final usa todo o treino e é avaliado na validação de 2025. O baseline D-7 repete a quantidade da mesma combinação curso/refeição de sete dias antes; é uma heurística proposta, não prática confirmada do RU. Tempos, parâmetros, métricas e previsões identificadas são salvos em dados/saidas/e06_bianca. Este bloco não implementa árvore/ensemble nem análise de erros.

Saídas: validacao_cruzada_ridge_por_dobra.csv; resumo_validacao_cruzada_ridge.csv; previsoes_validacao_bianca.csv; previsoes_total_ru_validacao_bianca.csv; registro_execucao_e06_bianca.csv.